# Topic Modelling: LDA, LSI and BERTopic

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandroviajando/ir/blob/wip/2_topic_modelling.ipynb)

This sesison will cover probabilistic and algebraic representations. You will be working with implementations of topic models using the [`gensim` library](https://radimrehurek.com/gensim/intro.html) and get an introduction to implementations of static and contextualised word embeddings.

In [ ]:
! pip install numpy scipy datasets scikit-learn gensim pyLDAvis BERTopic

import datasets
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

import gensim
import gensim.corpora as corpora
from gensim.utils import simple_preprocess
from gensim.models import CoherenceModel

import numpy as np
import pandas as pd
from pprint import pprint
import scipy
from sklearn.decomposition import TruncatedSVD
import tqdm

We will work with the simple English wikipedia again, as we did in the previous session. Load the dataset with the command `datasets.load_dataset("wikipedia", "20220301.simple", trust_remote_code=True)["train"]` (You can use a subset of 2k-5k articles for the exercise to reduce computation time).

In [2]:
wikipedia = datasets.load_dataset("wikipedia", "20220301.simple", trust_remote_code=True)["train"].select(range(200))

The first step to every text processing pipeline is the tokenisation and preprocessing. Define a preprocessing function that takes a string of text and tokenises, lemmatises and removes stopwords and returns a list of tokens for further analyses. You can either use only `nltk` or combine it with preprocessing functions defined in the `gensim` library, for eg. `gensim.utils.simple_preprocess()` for this task.

In [ ]:
stop_words = stopwords.words("english")
lemmatiser = nltk.stem.WordNetLemmatizer

def preprocess(doc):
    doc = gensim.utils.simple_preprocess(doc, deacc=True)
    doc = [word for word in doc if word not in stop_words]
    doc = [lemmatiser().lemmatize(word) for word in doc]
    return doc

In [ ]:
wikipedia = wikipedia.select(range(2000))
preprocessed_data = [preprocess(doc) for doc in wikipedia["text"]]

You can now use your preprocessed text to create your topic models. We will compare the gensim implementations of the LDA and LSI models by training them on the simple English wikipedia.

You can go through the documentation for the [LDAModel](https://radimrehurek.com/gensim/models/ldamodel.html) and the [LSIModel](https://radimrehurek.com/gensim/models/lsimodel.html). To get started with training your models, you will need the following variables (assuming `preprocessed_data` is a list of lists - with each containing list being the list of tokens in each document generated by your preprocess function):

In [ ]:
# Dictionary
id2word = corpora.Dictionary(preprocessed_data)

# Term Document Frequency Bag of Words representation of the document
corpus = [id2word.doc2bow(text) for text in preprocessed_data]

# List of documents with each tuple being (word_id, word count in doc)
print(corpus[:1])

[[(0, 2), (1, 1), (2, 1), (3, 1), (4, 1), (5, 2), (6, 1), (7, 1), (8, 2), (9, 1), (10, 3), (11, 1), (12, 1), (13, 1), (14, 3), (15, 1), (16, 2), (17, 1), (18, 1), (19, 5), (20, 2), (21, 1), (22, 1), (23, 3), (24, 1), (25, 1), (26, 1), (27, 1), (28, 208), (29, 1), (30, 1), (31, 1), (32, 1), (33, 1), (34, 2), (35, 4), (36, 3), (37, 1), (38, 1), (39, 1), (40, 2), (41, 1), (42, 1), (43, 1), (44, 2), (45, 2), (46, 1), (47, 1), (48, 2), (49, 1), (50, 1), (51, 1), (52, 1), (53, 1), (54, 1), (55, 1), (56, 1), (57, 2), (58, 1), (59, 13), (60, 1), (61, 5), (62, 1), (63, 1), (64, 1), (65, 1), (66, 1), (67, 1), (68, 1), (69, 1), (70, 1), (71, 9), (72, 2), (73, 1), (74, 1), (75, 1), (76, 1), (77, 2), (78, 1), (79, 1), (80, 1), (81, 2), (82, 1), (83, 1), (84, 2), (85, 1), (86, 1), (87, 1), (88, 1), (89, 1), (90, 1), (91, 1), (92, 1), (93, 1), (94, 1), (95, 3), (96, 2), (97, 1), (98, 1), (99, 2), (100, 1), (101, 2), (102, 1), (103, 2), (104, 1), (105, 1), (106, 1), (107, 1), (108, 2), (109, 1), (110,

Gensim creates a unique id for each word in the document. The produced corpus shown above is a mapping of `(word_id, word_frequency)`.

For example, `(0, 1)` above implies, word id `0` occurs once in the first document. Likewise, word id `1` occurs twice and so on.

This is used as the input by the LDA model.

If you want to see what word a given id corresponds to, pass the id as a key to the dictionary.

In [6]:
print(id2word[0])

abdicates


Or, you can see a human-readable form of the corpus (term frequency):

In [7]:
[[(id2word[id], freq) for id, freq in cp] for cp in corpus[:1]]

[[('abdicates', 2),
  ('abraham', 1),
  ('additionally', 1),
  ('adolf', 1),
  ('affecting', 1),
  ('africa', 2),
  ('aged', 1),
  ('aimed', 1),
  ('air', 2),
  ('albert', 1),
  ('alexander', 3),
  ('alois', 1),
  ('along', 1),
  ('alphabetical', 1),
  ('also', 3),
  ('alvares', 1),
  ('always', 2),
  ('ambedkar', 1),
  ('america', 1),
  ('american', 5),
  ('angola', 2),
  ('anne', 1),
  ('another', 1),
  ('anzac', 3),
  ('apart', 1),
  ('aperire', 1),
  ('aphrodite', 1),
  ('apple', 1),
  ('april', 208),
  ('aragon', 1),
  ('arbroath', 1),
  ('argentina', 1),
  ('argentine', 1),
  ('aries', 1),
  ('armenia', 2),
  ('army', 4),
  ('around', 3),
  ('arranged', 1),
  ('ascension', 1),
  ('ash', 1),
  ('asian', 2),
  ('assassination', 1),
  ('assistance', 1),
  ('astrological', 1),
  ('australia', 2),
  ('australian', 2),
  ('autism', 1),
  ('autumn', 1),
  ('awareness', 2),
  ('back', 1),
  ('baha', 1),
  ('bangladesh', 1),
  ('barbados', 1),
  ('baseball', 1),
  ('basque', 1),
  ('battl

You can now use these to build the LDA model:

In [8]:
lda_model = gensim.models.ldamodel.LdaModel(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=20,
                                           random_state=100,
                                           update_every=1,
                                           chunksize=100,
                                           passes=50,
                                           alpha="auto",
                                           per_word_topics=True)

Apart from that, `alpha` and `eta` are hyperparameters that affect sparsity of the topics. According to the Gensim docs, both default to 1.0/num_topics prior.
`chunksize` is the number of documents to be used in each training chunk. `update_every` determines how often the model parameters should be updated and `passes` is the total number of training passes.

You can try different settings to determine the ideal for your model. Try reducing or increasing the number of passes, how does that affect your model? Can you find the ideal number of topics?

The above LDA model is built with 20 different topics where each topic is a combination of keywords and each keyword contributes a certain weightage to the topic. You can examine the topics e.g. by printing the top 10 keywords in the topics as follows:

In [9]:
lda_model.show_topics(num_topics=20, num_words=10, formatted=False)

[(0,
  [('plague', 0.022763794),
   ('disease', 0.020207478),
   ('bacteria', 0.011882668),
   ('oral', 0.009727187),
   ('form', 0.008477872),
   ('cup', 0.008427534),
   ('flea', 0.008427274),
   ('pandemic', 0.008427274),
   ('colchester', 0.0084271),
   ('spread', 0.008375806)]),
 (1,
  [('country', 0.020225493),
   ('people', 0.017653717),
   ('state', 0.01208022),
   ('many', 0.010113256),
   ('government', 0.009337602),
   ('world', 0.009281852),
   ('city', 0.00833562),
   ('war', 0.007877211),
   ('capital', 0.007829203),
   ('island', 0.007397676)]),
 (2,
  [('galaxy', 0.033260852),
   ('star', 0.01858303),
   ('comedy', 0.012730466),
   ('comet', 0.011929691),
   ('astronomy', 0.011189222),
   ('ecology', 0.009994902),
   ('type', 0.009410692),
   ('biology', 0.009291006),
   ('thing', 0.0077095125),
   ('cheese', 0.0076443036)]),
 (3,
  [('god', 0.041130863),
   ('people', 0.032047186),
   ('person', 0.02509928),
   ('human', 0.024903689),
   ('body', 0.024638604),
   ('rel

Topic Coherence is a convenient measure to judge how good a given topic model is. While not an absolute indicator of quality, topic coherence can give you a general idea of how 'coherent' your model is by assigning a score between 0 and 1. You can learn more here:

https://developer.ibm.com/tutorials/awb-lda-topic-modeling-text-analysis-python/#step-6-evaluate-models8

Gensim has an implementation of topic coherence we can use to evaluate our model:

In [10]:
coherence_model_lda = CoherenceModel(model=lda_model, texts=preprocessed_data, dictionary=id2word, coherence="c_v")
coherence_lda = coherence_model_lda.get_coherence()
print("\nCoherence Score: ", coherence_lda)


Coherence Score:  0.40999058086828766


Exclusively for LDA models (this does not work for LSI), you can visualise and interact with the topics your model creates using the pyLDAvis package’s interactive chart that is designed to work well with jupyter notebooks:

In [11]:
import pyLDAvis
import pyLDAvis.gensim
import matplotlib.pyplot as plt
%matplotlib inline

In [12]:
pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
vis

PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
1      0.192476  0.166057       1        1  22.545380
13     0.237602 -0.065554       2        1  14.952319
9      0.172171  0.069254       3        1  14.073333
8      0.224082 -0.055365       4        1  13.997399
3      0.141751 -0.109588       5        1   4.809986
11    -0.020094  0.289158       6        1   4.563026
7      0.096923 -0.091573       7        1   4.198153
2      0.089781 -0.073509       8        1   4.183920
6      0.009796 -0.059713       9        1   3.563876
15    -0.024566  0.212604      10        1   3.173398
17    -0.035278 -0.074252      11        1   2.083473
19    -0.055704 -0.054435      12        1   1.884775
14    -0.090748  0.029771      13        1   1.583875
16    -0.108409 -0.027483      14        1   1.149690
4     -0.116248 -0.028266      15        1   1.002530
10    -0.097935 -0.037750      16        1   0.902265
12    -0.169266 -0.024960      17        1   0.795845
0     -0.149700 -0.031014      18        1   0.382631
5     -0.147338 -0.020292      19        1   0.152016
18    -0.149296 -0.013089      20        1   0.002111, topic_info=          Term        Freq       Total Category  logprob  loglift
160        day  342.000000  342.000000  Default  30.0000  30.0000
127       city  213.000000  213.000000  Default  29.0000  29.0000
28       april  180.000000  180.000000  Default  28.0000  28.0000
704       year  354.000000  354.000000  Default  27.0000  27.0000
228   february  168.000000  168.000000  Default  26.0000  26.0000
..         ...         ...         ...      ...      ...      ...
25     aperire    0.000143    1.540417  Topic20  -9.2656   1.4782
26   aphrodite    0.000143    1.540417  Topic20  -9.2656   1.4782
27       apple    0.000143   94.053816  Topic20  -9.2656  -2.6336
28       april    0.000143  180.889270  Topic20  -9.2656  -3.2877
29      aragon    0.000143    5.974929  Topic20  -9.2656   0.1227

[1187 rows x 6 columns], token_table=      Topic      Freq          Term
term                               
2689      6  0.930636  abbreviation
0         6  0.312981     abdicates
0        10  0.625961     abdicates
4815      3  0.080276       ability
4815      5  0.883031       ability
...     ...       ...           ...
704      11  0.002822          year
704      13  0.005644          year
5192     18  0.717805      yersinia
5989     14  0.797334     zealandia
5193     18  0.563817      zoonosis

[2425 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[2, 14, 10, 9, 4, 12, 8, 3, 7, 16, 18, 20, 15, 17, 5, 11, 13, 1, 6, 19])

Each bubble on the left-hand side plot represents a topic. The larger the bubble, the more prevalent is that topic.

- A good topic model will have fairly big, non-overlapping bubbles scattered throughout the chart instead of being clustered in one quadrant.
- A model with too many topics will typically have many overlaps, small sized bubbles clustered in one region of the chart.
- If you move the cursor over one of the bubbles, the words and bars on the right-hand side will update. These words are the salient keywords that form the selected topic.

Does it look like you have a good model based on the visualisation?

**Task:** Train an LSI model (`gensim.models.lsimodel.LsiModel()`) on the simple English wikipedia and compute the coherence score. How does it compare to the coherence score of the LDA model?

In [ ]:
from gensim.models.lsimodel import LsiModel

lsi_model = LsiModel(corpus=corpus, id2word=id2word, num_topics=20)
lsi_model.show_topics(num_topics=20, num_words=10, formatted=False)

/opt/homebrew/anaconda3/envs/ir/lib/python3.11/site-packages/gensim/models/lsimodel.py:963: DeprecationWarning: `scipy.sparse.sparsetools.csc_matvecs` is deprecated along with the `scipy.sparse.sparsetools` namespace. `scipy.sparse.sparsetools.csc_matvecs` will be removed in SciPy 1.14.0, and the `scipy.sparse.sparsetools` namespace will be removed in SciPy 2.0.0.
  sparsetools.csc_matvecs(


[(0,
  [('day', -0.4976204079783755),
   ('april', -0.47553651371528166),
   ('year', -0.29856234842084267),
   ('february', -0.21562048964183578),
   ('people', -0.18193098123627832),
   ('december', -0.1767732383665591),
   ('august', -0.13896034629038245),
   ('country', -0.10648170519034737),
   ('first', -0.10643005950039688),
   ('world', -0.09586333677337583)]),
 (1,
  [('april', 0.35864041940383323),
   ('people', -0.3073602579471891),
   ('day', 0.2787069237990462),
   ('computer', -0.2670490719664267),
   ('many', -0.198562567187497),
   ('country', -0.16258358942556664),
   ('also', -0.14948274867456646),
   ('one', -0.14843122463282915),
   ('called', -0.1471925181349259),
   ('australia', -0.13460946242993496)]),
 (2,
  [('april', 0.6858480194581439),
   ('february', -0.48691337031797527),
   ('august', -0.27357249852097393),
   ('december', -0.2604931239892763),
   ('day', -0.1733510303960889),
   ('year', -0.16267851019660876),
   ('computer', 0.1419494241702293),
   ('i

In [14]:
coherence_model_lsi = CoherenceModel(model=lsi_model, texts=preprocessed_data, dictionary=id2word, coherence="c_v")
coherence_lsi = coherence_model_lsi.get_coherence()
print("\nCoherence Score: ", coherence_lsi)


Coherence Score:  0.37369054564328785


You can also assign topics to unseen documents (a "query" in the IR context) using your models. In the retrieval context, you can find relevant documents for a query by finding the closest topic to the query, and then ranking salient documents in that topic.

In [ ]:
query = "computers and technology"
query_bow = id2word.doc2bow(preprocess(query)) # doc to bag of words representation
query_lda = lda_model[query_bow] # vector representation of the query
query_lsi = lsi_model[query_bow]

In [17]:
if query_lda and all(isinstance(item, (list, tuple)) and len(item) >= 2 for item in query_lda):
    closest_topic_lda = max(query_lda[0], key=lambda x: x[1])[0]
    print(f"The closest topic is {closest_topic_lda}")
    lda_model.show_topic(closest_topic_lda)
else:
    # Handle the case where query_lda is empty or invalid
    print("LDA model did not find any relevant topics for the query.")
    closest_topic_lda = None # Or assign a default topic

The closest topic is 3


In [18]:
if query_lsi and all(isinstance(item, (list, tuple)) and len(item) >= 2 for item in query_lsi):
    closest_topic_lsi = max(query_lsi, key=lambda x: x[1])[0]
    print(f"The closest topic is {closest_topic_lsi}")
    lsi_model.show_topic(closest_topic_lsi)
else:
    # Handle the case where query_lda is empty or invalid
    print("LSI model did not find any relevant topics for the query.")
    closest_topic_lsi = None # Or assign a default topic

The closest topic is 14


We can use the following functions to retrieve relevant wikipedia articles for a given topic (since the implementations for LDA and LSI differ, we will define separate functions for each):

In [22]:
def get_documents_for_topic_lda(topic_id, num_docs=5):
    # Get the topic distribution for each document in the corpus
    document_topic_probs = [lda_model.get_document_topics(doc) for doc in corpus]

    # Find documents where the specified topic is most dominant
    relevant_documents_indices = []
    for i, doc_topics in enumerate(document_topic_probs):
        # Find the topic with the highest probability for this document
        dominant_topic = max(doc_topics, key=lambda item: item[1])

        # If the dominant topic matches the specified topic_id, add the document index
        if dominant_topic[0] == topic_id:
            relevant_documents_indices.append(i)

    # Extract the corresponding document texts from the wikipedia dataset
    relevant_documents = [wikipedia["url"][i] for i in relevant_documents_indices]

    return relevant_documents[:num_docs]

relevant_documents = get_documents_for_topic_lda(closest_topic_lda)
for i, doc in enumerate(relevant_documents):
    print(f"Document {i + 1}: {doc}")

Document 1: https://simple.wikipedia.org/wiki/Ad%20hominem
Document 2: https://simple.wikipedia.org/wiki/Abrahamic%20religion
Document 3: https://simple.wikipedia.org/wiki/Anatomy
Document 4: https://simple.wikipedia.org/wiki/Being
Document 5: https://simple.wikipedia.org/wiki/Creator


In [23]:
def get_documents_for_topic_lsi(topic_id, num_docs=5):
    # Get topic weights for all documents
    doc_topic_dist = np.array([np.array(lsi_model[doc])[:, 1] for doc in corpus])

    # Get document indices sorted by weight for the given topic
    sorted_doc_indices = np.argsort(doc_topic_dist[:, topic_id], axis=0)[::-1]

    # Extract document URLs for the top documents
    relevant_documents = [wikipedia["url"][i] for i in sorted_doc_indices[:num_docs]]

    return relevant_documents

relevant_documents = get_documents_for_topic_lsi(closest_topic_lsi)
for i, doc_url in enumerate(relevant_documents):
    print(f"Document {i + 1}: {doc_url}")

Document 1: https://simple.wikipedia.org/wiki/Fish
Document 2: https://simple.wikipedia.org/wiki/Afghanistan
Document 3: https://simple.wikipedia.org/wiki/Denmark
Document 4: https://simple.wikipedia.org/wiki/Apple
Document 5: https://simple.wikipedia.org/wiki/Hair


**Task:** Compare the documents returned by the LDA and LSI model on the following queries (same as last session) for a more qualitative analysis of the two models:

1. Earth's atmosphere
2. Agricultural crops
3. Parts of the human body
4. What are the official languages of countries?
5. Best places to travel

You can also experiment with your own queries. Which model do you think is better? And why?



In [ ]:
# your code here

### Bonus: BERTopic

BERTopic uses more advanced neural representations of transformer models and c-TF-IDF for topic modelling. A bried overview of the algorithm can be found here: https://maartengr.github.io/BERTopic/algorithm/algorithm.html. To understand in more detail, please see the introductory paper by [Grootendorst(2022)](https://arxiv.org/pdf/2203.05794).

The `BERTopic` library provides an easy way to create topic models with BERT.

**Task:** Go through the `BERTopic` quickstart documentation and try out the library. Train a BERTopic model on the simple English Wikipedia, compute the coherence score and compare it to the LDA and LSI models. Run the above queries with the BERTopic model and observe the retrieved documents. Does BERTopic do better than LDA and LSI?

In [ ]:
from bertopic import BERTopic

bertopic_model = BERTopic()
topics, probs = bertopic_model.fit_transform(wikipedia["text"])

In [ ]:
similar_topics, similarity = bertopic_model.find_topics("parts of a human body", top_n=5)

print(similar_topics)


Coherence Score:  1.0


## **Introduction to word embeddings**

You must have noticed that in the three topic models we worked with, we used different types of word representations. The LSI model uses word embeddings obtained by applying SVD dimensionality reduction on a term-document frequency matrix. BERTopic, on the other hand, uses neural word embeddings learnt by a transformer model.

Let's understand this a bit further.
We will start with a sparse matrix, which we will perform some kind of dimensionality reduction on to create dense representatins. We will build a cooccurance matrix encompassing how often individual words co-occur with all other words in the vocabulary. This should be a symmetric matrix with a dimensionality of $|V| \times |V|$, where $|V|$ is the vocabulary size.

In [34]:
vocabulary = {}
data, row, col = [], [], []

window_size = 1  # the context window to calculate co-occurance - this can be adjusted

for tokens in tqdm.tqdm(preprocessed_data, desc="Generating vocabulary and computing coo matrix from documents."):
    for pos, token in enumerate(tokens):
        i = vocabulary.setdefault(token, len(vocabulary))
        start = max(0, pos - window_size)
        end = min(len(tokens), pos + window_size + 1)
        for pos2 in range(start, end):
            if pos2 == pos:
                continue
            j = vocabulary.setdefault(tokens[pos2], len(vocabulary))
            data.append(1.)
            row.append(i)
            col.append(j)

cooccurrence_matrix = scipy.sparse.coo_matrix((data, (row, col)))
idx_to_word = {idx: word for word, idx in vocabulary.items()}

Generating vocabulary and computing coo matrix from documents.: 100%|██████████| 200/200 [00:00<00:00, 3555.41it/s]


In [35]:
df = pd.DataFrame(cooccurrence_matrix.toarray(), index=vocabulary.keys(), columns=vocabulary.keys())
df

,april,fourth,month,year,julian,gregorian,calendar,come,march,may,...,pint,pt,quart,ounce,arises,mm,fl,oz,litre,iso
april,6.0,1.0,1.0,12.0,0.0,0.0,0.0,1.0,10.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
fourth,1.0,0.0,2.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
month,1.0,2.0,4.0,12.0,0.0,0.0,2.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
year,12.0,1.0,12.0,38.0,3.0,4.0,2.0,0.0,4.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
julian,0.0,0.0,0.0,3.0,0.0,2.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
mm,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
fl,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0
oz,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0
litre,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [36]:
SVD = TruncatedSVD(n_components=100, n_iter=10, random_state=42)
reduced = SVD.fit_transform(cooccurrence_matrix)

print(reduced.shape)

(10568, 100)


You can examine the quality of your embeddings by comparing the embeddings for similar words:

In [37]:
def most_similar(word):
    word_vector = reduced[vocabulary[word]]
    ranked_results = np.argsort(reduced @ word_vector.T)[::-1][:1]

    most_similar = [idx_to_word[i] for i in ranked_results][0]

    return most_similar

print(f"Most similar word to human is {most_similar('human')}")

Most similar word to human is human


**Task:**

1. Try this for different words. Does your implementation return relevant results? If not, can you determine why?
2. The preprocessing step can have a big impact on the values here. Try adding in stopwords, or not doing lemmatisation or lower casing - what does that do to your results here? Does it give you better embeddings?
3. Evaluate your model on the analogies dataset, which you can load via `datasets.load_dataset("tomasmcz/word2vec_analogy")`. This dataset was introduced in [one of the original `word2vec` papers](https://arxiv.org/pdf/1310.4546.pdf), highlighting the ability of word embeddings to capture the semantic orientation of the vocabulary. For example, given an analogy such as _Berlin is to Germany as Paris is to_ ____ _?_, it is possible to perform simple vector arithmetic to retrieve the answer _France_:
$$\vec{\mathrm{Germany}} - \vec{\mathrm{Berlin}} + \vec{\mathrm{Paris}} \approx \vec{\mathrm{France}}$$
Apply this formula to the dataset, keeping track of cases where your model retrieves the correct answer. What is its overall accuracy? How many OOV tokens do you observe? (Not: you can use simple dot product similarity to compare your vectors)
4. Experiment with neural network-based word embeddings such as `word2vec` or `GLoVe`. You can load these from the gensim library. Try to calculate how well these models fare on the analogies dataset. Do they perform or worse than your model?

In [38]:
analogies = datasets.load_dataset("tomasmcz/word2vec_analogy")["train"]
analogies

README.md:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

capital-common-countries_001.csv:   0%|          | 0.00/14.7k [00:00<?, ?B/s]

capital-world_001.csv:   0%|          | 0.00/142k [00:00<?, ?B/s]

city-in-state_001.csv:   0%|          | 0.00/85.7k [00:00<?, ?B/s]

currency_001.csv:   0%|          | 0.00/22.9k [00:00<?, ?B/s]

...

gram7-past-tense_001.csv:   0%|          | 0.00/47.7k [00:00<?, ?B/s]

gram8-plural_001.csv:   0%|          | 0.00/34.4k [00:00<?, ?B/s]

gram9-plural-verbs_001.csv:   0%|          | 0.00/24.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19544 [00:00<?, ? examples/s]

Dataset({
    features: ['word_a', 'word_b', 'word_c', 'word_d'],
    num_rows: 19544
})

In [ ]:
analogies["word_a"][0], analogies["word_b"][0], analogies["word_c"][0], analogies["word_d"][0]

In [ ]:
word1 = analogies["word_a"][8756].lower()
word2 = analogies["word_b"][8756].lower()
word3 = analogies["word_c"][8756].lower()
word4 = analogies["word_d"][8756].lower()  # Correct answer
print(word1, word2, word3, word4)

embedding1 = reduced[vocabulary[word1]]
embedding2 = reduced[vocabulary[word2]]
embedding3 = reduced[vocabulary[word3]]

if all(embedding is not None for embedding in [embedding1, embedding2, embedding3]):
    predicted_embedding = embedding2 - embedding1 + embedding3
    # Calculate cosine similarity with all words in the vocabulary
    similarities = {}
    for word, index in vocabulary.items():
            embedding4 = reduced[index]
            similarity = np.dot(predicted_embedding, embedding4) / (np.linalg.norm(predicted_embedding) * np.linalg.norm(embedding4))
            similarities[word] = similarity

    # Get the word with the highest similarity
    predicted_word = max(similarities, key=similarities.get)

print(predicted_word)

In [ ]:
import gensim.downloader as api

# Load Google's pre-trained Word2Vec and a GloVe model (300-dimensional vectors)
word2vec = api.load("word2vec-google-news-300")
glove = api.load("glove-wiki-gigaword-300")

# Get the vector for a word
word = "king"
vector = word2vec[word]

print(f"Vector for '{word}':\n", vector)
print("Vector shape:", vector.shape)

In [ ]:
# Model vocab can be accessed through:
word2vec.index_to_key

In [ ]:
word1 = analogies["word_a"][8756].lower()
word2 = analogies["word_b"][8756].lower()
word3 = analogies["word_c"][8756].lower()
word4 = analogies["word_d"][8756].lower()  # Correct answer
print(word1, word2, word3, word4)

embedding1 = word2vec[word1]
embedding2 = word2vec[word2]
embedding3 = word2vec[word3]

if all(embedding is not None for embedding in [embedding1, embedding2, embedding3]):
    predicted_embedding = embedding2 - embedding1 + embedding3
    # Calculate cosine similarity with all words in the vocabulary
    similarities = {}
    for word in word2vec.index_to_key:
            embedding4 = word2vec[word]
            similarity = np.dot(predicted_embedding, embedding4) / (np.linalg.norm(predicted_embedding) * np.linalg.norm(embedding4))
            similarities[word] = similarity

    # Get the word with the highest similarity
    predicted_word = max(similarities, key=similarities.get)

print(predicted_word)